# LNL on GTSRB — Nhóm X — **kiểm chứng độ chính xác (plug-and-play)**

Notebook này để **thầy kiểm chứng** kết quả Top-1 của nhóm mà **không cần train lại**.
Nó làm đúng quy trình plug-and-play:

1. clone repo của nhóm (đã chứa `LNL.py` cải tiến);
2. tải **trọng số đã train sẵn** (`lnl_gtsrb.pth`) từ GitHub Release của repo;
3. dựng model **y hệt** `Instructions.ipynb` gốc: `LNL_Ti` → `head = Linear(192, 43)`;
4. nạp trọng số rồi chạy **đúng cell Test gốc** (ảnh thô `[0,1]`, không augmentation, không TTA).

> Chuẩn hoá ImageNet được nhúng **bên trong** `LNL.py` (`forward_features`), nên cell Test gốc đưa
> ảnh thô `[0,1]` vào là khớp — không cần sửa gì ở notebook.

**Cách chạy:** Runtime → Change runtime type → **GPU (T4)** → **Runtime → Run all**.
Số cần báo cáo là **`Standard accuracy`** ở mục 6.

## 1. GPU + clone repo nhóm + cài deps

In [ ]:
!nvidia-smi -L
%cd /content
!rm -rf LNL-GTSRB-NhomX
!git clone -q https://github.com/bnqtoan/LNL-GTSRB-NhomX.git
%cd /content/LNL-GTSRB-NhomX
!pip install -q timm einops torchattacks 2>&1 | tail -1
print('repo + deps ready')

## 2. Tải trọng số nhóm (`lnl_gtsrb.pth`) từ GitHub Release
Tải tự động từ `https://github.com/bnqtoan/LNL-GTSRB-NhomX/releases/download/weights-v1/lnl_gtsrb.pth`.
Nếu vì lý do mạng không tải được, có thể tải tay file `.pth` nhóm nộp rồi upload vào
`/content/LNL-GTSRB-NhomX/` với đúng tên `lnl_gtsrb.pth`.

In [ ]:
import os
WEIGHTS = '/content/LNL-GTSRB-NhomX/lnl_gtsrb.pth'
URL = 'https://github.com/bnqtoan/LNL-GTSRB-NhomX/releases/download/weights-v1/lnl_gtsrb.pth'
if not os.path.exists(WEIGHTS):
    !curl -L -o $WEIGHTS $URL
assert os.path.exists(WEIGHTS) and os.path.getsize(WEIGHTS) > 1_000_000, \
    'Không tải được trọng số — hãy upload tay lnl_gtsrb.pth vào thư mục repo.'
print('weights:', WEIGHTS, '| %.1f MB' % (os.path.getsize(WEIGHTS)/1e6))

## 3. Tải + chuẩn bị GTSRB (như `Instructions.ipynb` gốc)

In [ ]:
import os, shutil
!mkdir -p data
base='https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370'
if not os.path.exists('data/GTSRB/Final_Test/Images'):
    !curl -sL $base/GTSRB_Final_Test_Images.zip -o data/test.zip
    !curl -sL $base/GTSRB_Final_Test_GT.zip     -o data/test_gt.zip
    !unzip -q -o data/test.zip    -d data/
    !unzip -q -o data/test_gt.zip -d data/
data_dir='./data/GTSRB'; images_dir=os.path.join(data_dir,'Final_Test/Images')
test_dir=os.path.join(data_dir,'test'); os.makedirs(test_dir, exist_ok=True)
if len(os.listdir(test_dir)) < 43:
    with open('./data/GT-final_test.csv') as f: rows=f.readlines()
    for t in rows[1:]:
        cls=int(t.split(';')[-1]); name=t.split(';')[0]
        d=os.path.join(test_dir, f'{cls:04d}'); os.makedirs(d, exist_ok=True)
        shutil.copy(os.path.join(images_dir, name), d)
print('GTSRB test ready | classes:', len(os.listdir(test_dir)))

## 4. DataLoader test — **y hệt notebook gốc** (Resize + ToTensor, ảnh thô `[0,1]`)

In [ ]:
import torch, torchvision
import torchvision.transforms as transforms
batch_size = 15
testset = torchvision.datasets.ImageFolder(root='./data/GTSRB/test',
            transform=transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()]))
test_loader = torch.utils.data.DataLoader(dataset=testset, batch_size=batch_size, shuffle=True)
print('test images:', len(testset))

## 5. Dựng model **y hệt gốc** rồi nạp trọng số
`LNL_Ti(pretrained=False)` → `model.head = Linear(192, 43)` → `.cuda()` → `load_state_dict`.

In [ ]:
from LNL import LNL_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()
missing, unexpected = model.load_state_dict(torch.load(WEIGHTS, map_location='cuda'), strict=False)
print('missing keys :', missing)
print('unexpected   :', unexpected)
assert not missing and not unexpected, 'STATE_DICT MISMATCH — trọng số không khớp model!'
print('Đã nạp trọng số — sẵn sàng Test.')

## 6. Test — **đúng cell Test gốc** → đây là số cần báo cáo

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

## 7. (Tuỳ chọn) FGSM / PGD — giữ nguyên từ notebook gốc
Không bắt buộc để chấm Top-1; để đây cho đầy đủ như notebook gốc.

In [ ]:
from torchattacks import FGSM
model.eval(); correct = 0; total = 0
atk = FGSM(model, eps=0.01)
for images, labels in test_loader:
    images = atk(images, labels).cuda()
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
print('Robust accuracy (FGSM eps=0.01): %.2f %%' % (100 * float(correct) / total))

In [ ]:
from torchattacks import PGD
model.eval(); correct = 0; total = 0
atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)
for images, labels in test_loader:
    images = atk(images, labels).cuda()
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
print('Robust accuracy (PGD eps=0.01): %.2f %%' % (100 * float(correct) / total))